# Import Library

In [ ]:
#Initial setup
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display

import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.linear_model import LogisticRegression
import urllib.request
import joblib

# Import dataset

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_val_Assay.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data = temp
display(pro_data)

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_dis_Assay.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data_dis = temp
display(pro_data_dis)

In [ ]:
fileDir = "./Data/"
fileName = "pro_sample_val.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
temp["icdo3m"] = temp["icdo3m"].astype("category")

pro_sample = temp
display(pro_sample)

In [ ]:
fileDir = "./Data/"
fileName = "map_OID_Uni_Sym.csv"

path = os.path.join(fileDir, fileName)
IDdata = pd.read_csv(path)
IDdata.set_index(IDdata.columns[0],inplace=True)
IDdata.index.name = None
display(IDdata)

In [ ]:
fileDir = "./Data/"
fileName = "DEP_list.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
display(temp)
DEPlist = temp

In [ ]:
fileDir = "./Data/"
fileName = "pro_data_common_dis.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_data_dis= temp
display(pro_data_dis)

In [ ]:
fileDir = "../Proteomics/Data/"
fileName = "CA199.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
temp = temp.sort_values('SampleID')
temp = temp.set_index('SampleID')
ca199_dis= temp
display(ca199_dis)

In [ ]:
fileDir = "./Data/"
fileName = "pro_sample_dis.csv"

path = os.path.join(fileDir, fileName)
temp = pd.read_csv(path)
temp.set_index(temp.columns[0],inplace=True)
temp.index.name = None
pro_sample_dis = temp
display(pro_sample_dis)

# モデル学習

log変換、欠損値の処理、標準化済み

## ①全プロテオミクスver

In [ ]:
target_sites = ["C250", "C251", "C252"]
mask = pro_sample["icdo3t"].astype(str).isin(target_sites)

X = pro_data.loc[mask, :]
y = pro_sample["icdo3t"].loc[mask].astype(str)

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
import time

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    param_grid = {
        'n_estimators': [100, 300, 500],
        "max_depth": [None, 5, 10, 15],
        # 'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [None, 'sqrt'],
        'class_weight': ['balanced']
    }

    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        # class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc_ovr',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    start = time.time()
    # 訓練の実行
    modelcv.fit(X_train, y_train)
    elapsed = time.time() - start
    print(f"1回のGridSearchCV: {elapsed / 60:.2f} 分")
    print(f"全5 foldの推定時間: {elapsed * 5 / 60:.2f} 分")

    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)
    
    score = roc_auc_score(
        y_test,
        y_prob,
        multi_class="ovr",
        average="macro"
    )
    aucs.append(score)

    print('AUC: ', score)
    
    # RocCurveDisplay.from_predictions(
    #     y_test,
    #     y_prob,
    #     name=f"Fold {i+1}", ax=ax
    # )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()



importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": model.feature_importances_
})
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)
print(importance_df.head(20))



In [ ]:
saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_allpro.joblib'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

In [ ]:

tempA = model.predict(X_test)
tempS = pd.Series(tempA, index = X_test.index, name = 'PredictedClass')
dataframe = y_test.rename('TrueClass')
dataframe = pd.merge(tempS, dataframe, left_index = True, right_index = True, how = 'left')

dataframe2 = pro_data.copy()
display(dataframe['TrueClass'].value_counts())
display(dataframe['PredictedClass'].value_counts())
display(dataframe.loc[dataframe['TrueClass']!=dataframe['PredictedClass']])


## ②DEP listのもののみver

In [ ]:
display(DEPlist)

In [ ]:
X = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]
y = pro_sample["CaCo_Status"]

print(DEPlist.shape)
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
modelsets_HER_DEP = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300,400, 500],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3,5, 7],
        'min_samples_leaf': [1, 2, 3,4]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    modelsets_HER_DEP.append(model)


    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

importance_df = pd.DataFrame({
    "Protein": X.columns,
    "Importance": model.feature_importances_
})
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)
print(importance_df.head(20))

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_DEPlist'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

### 120人の方で検証

In [ ]:
X = pro_data_dis.loc[:, pro_data_dis.columns.isin(DEPlist.index)]

y = pro_sample_dis['groupset2']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
val_probs = []
val_aucs = []

fig, ax = plt.subplots()

for i, model in enumerate(modelsets_HER_DEP):

    y_val = model.predict_proba(X)[:, 1]
    val_probs.append(y_val)

    auc_val = roc_auc_score(y, y_val)
    val_aucs.append(auc_val) 

    print(f"Fold {i+1} Validation AUC: {auc_val:.4f}")

    RocCurveDisplay.from_predictions(
        y,
        y_val,
        name=f"Fold {i+1} (AUC = {auc_val:.3f})",
        ax=ax
    )

# 5モデルの予測確率を平均
val_probs = np.array(val_probs)
y_val_mean = val_probs.mean(axis=0)

# Ensemble AUC
auc_mean = roc_auc_score(y, y_val_mean)

# 平均予測のROC
RocCurveDisplay.from_predictions(
    y,
    y_val_mean,
    name=f"Mean prediction (AUC = {auc_mean:.3f})",
    ax=ax
)
# 5モデルのAUCの平均と標準偏差
auc_val_mean = np.mean(val_aucs)
auc_val_sd = np.std(val_aucs, ddof=1)

print(f"Validation AUC: {auc_val_mean:.4f} ± {auc_val_sd:.4f}")

ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_title("Validation ROC curves")
plt.show()


print(f"Ensemble Validation AUC: {auc_mean:.4f}")

# CA-19-9とIL2RAのモデル

## CA-19-9とIL2RAのデータ読み込み

In [ ]:
ca199 = pro_sample[['CA19_9']].copy()
display(ca199)

il2ra = pro_sample[['sIL2R']].copy()
display(il2ra)

In [ ]:
print((ca199['CA19_9'] <= 0).sum())
print((il2ra['sIL2R'] <= 0).sum())

In [ ]:
ca199['logvalue'] = np.log10(ca199['CA19_9'])
il2ra['logvalue'] = np.log10(il2ra['sIL2R'])


## CA-19-9単独

In [ ]:
X = ca199[["logvalue"]]
y = pro_sample["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
modelsets_HER_ca199 = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300,400, 500],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3,5, 7],
        'min_samples_leaf': [1, 2, 3,4]
    }
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    modelsets_HER_ca199.append(model)

    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_ca199'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

### 120人の方で検証

In [ ]:
X = ca199_dis[['logvalue']]

y = pro_sample_dis['groupset2']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
val_probs = []
val_aucs = []

fig, ax = plt.subplots()

for i, model in enumerate(modelsets_HER_ca199):

    y_val = model.predict_proba(X)[:, 1]
    val_probs.append(y_val)

    auc_val = roc_auc_score(y, y_val)
    val_aucs.append(auc_val)

    print(f"Fold {i+1} Validation AUC: {auc_val:.4f}")

    RocCurveDisplay.from_predictions(
        y,
        y_val,
        name=f"Fold {i+1} (AUC = {auc_val:.3f})",
        ax=ax
    )

# 5モデルの予測確率を平均
val_probs = np.array(val_probs)
y_val_mean = val_probs.mean(axis=0)

# Ensemble AUC
auc_mean = roc_auc_score(y, y_val_mean)

# 平均予測のROC
RocCurveDisplay.from_predictions(
    y,
    y_val_mean,
    name=f"Mean prediction (AUC = {auc_mean:.3f})",
    ax=ax
)

# 5モデルのAUCの平均 ± SD
auc_val_mean = np.mean(val_aucs)
auc_val_sd = np.std(val_aucs, ddof=1)

print(f"Validation AUC: {auc_val_mean:.4f} ± {auc_val_sd:.4f}")

ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_title("Validation ROC curves")
plt.show()

print(f"Ensemble Validation AUC: {auc_mean:.4f}")

## IL2RA単独

In [ ]:
X = il2ra[["logvalue"]]
y = pro_sample["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 3]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)
    
    model = modelcv.best_estimator_
    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_il2ra'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

## DEPlist + CA19-9

In [ ]:
dep = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]
X = pd.concat([
    ca199[['logvalue']],
    dep
], axis=1)

y = pro_sample['CaCo_Status']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
modelsets_HER_DEP_ca199 = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300,400, 500],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3,5, 7],
        'min_samples_leaf': [1, 2, 3,4]
    }
    
    model = RandomForestClassifier(
        criterion='gini',
        random_state=123,
        class_weight='balanced',
        n_jobs=-1 
    )
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    dataframe = pd.DataFrame(modelcv.cv_results_).sort_values('mean_test_score', ascending=False)
    display(dataframe)

    # 保存

    model = modelcv.best_estimator_
    modelsets_HER_DEP_ca199.append(model)

    y_prob = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()

saved_assets = {
    'model': modelcv.best_estimator_
}
name = 'mdl_DEPlist+ca199'
save_dir = "./Data"
path = os.path.join(save_dir, name)
joblib.dump(saved_assets, path)

### 120人の方で検証

In [ ]:
display(pro_data_dis)

In [ ]:
dep = pro_data_dis.loc[:, pro_data_dis.columns.isin(DEPlist.index)]
X = pd.concat([
    ca199_dis[['logvalue']],
    dep
], axis=1)

y = pro_sample_dis['groupset2']
print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
display(X)

In [ ]:
val_probs = []
val_aucs = []

fig, ax = plt.subplots()

for i, model in enumerate(modelsets_HER_DEP_ca199):

    y_val = model.predict_proba(X)[:, 1]
    val_probs.append(y_val)

    auc_val = roc_auc_score(y, y_val)
    val_aucs.append(auc_val)

    print(f"Fold {i+1} Validation AUC: {auc_val:.4f}")

    RocCurveDisplay.from_predictions(
        y,
        y_val,
        name=f"Fold {i+1} (AUC = {auc_val:.3f})",
        ax=ax
    )

# 5モデルの予測確率を平均
val_probs = np.array(val_probs)
y_val_mean = val_probs.mean(axis=0)

# Ensemble AUC
auc_mean = roc_auc_score(y, y_val_mean)

# 平均予測のROC
RocCurveDisplay.from_predictions(
    y,
    y_val_mean,
    name=f"Mean prediction (AUC = {auc_mean:.3f})",
    ax=ax
)

# 5モデルのAUCの平均と標準偏差
auc_val_mean = np.mean(val_aucs)
auc_val_sd = np.std(val_aucs, ddof=1)

print(f"Validation AUC: {auc_val_mean:.4f} ± {auc_val_sd:.4f}")

ax.plot([0, 1], [0, 1], linestyle="--")

ax.set_title("Validation ROC curves")
plt.show()

print(f"Ensemble Validation AUC: {auc_mean:.4f}")

## CA-19-9とIL2RAの複合

In [ ]:

X = pd.concat(
    [
        ca199[["logvalue"]].rename(columns={"logvalue": "ca199_logvalue"}),
        il2ra[["logvalue"]].rename(columns={"logvalue": "il2ra_logvalue"})
    ],
    axis=1
)
y = pro_sample["CaCo_Status"]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

### 2つの変数の「差」や「和」を新たな特徴量として加える

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
aucs = []
fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(class_weight='balanced', random_state=123, max_iter=1000))
    ])
    
    param_grid = {
        'classifier__C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0], # 正則化の強さ（小さいほど強力に過学習を防ぐ）
        'classifier__penalty': ['l2'] # Ridge正則化（変数を程よく残す設定）
    }
    
   
    
    cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        pipe, 
        param_grid=param_grid,
        scoring='roc_auc',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    modelcv.fit(X_train, y_train)
    
    print(f'--- Fold {i+1} ---')
    print('Best score: ', modelcv.best_score_)
    print('Best hyperparameters: ', modelcv.best_params_)
    
    best_pipe = modelcv.best_estimator_
    coefficients = best_pipe.named_steps['classifier'].coef_[0]
    print(f"ca199の重み: {coefficients[0]:.4f}")
    print(f"il2raの重み: {coefficients[1]:.4f}")
    
    y_prob = best_pipe.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_prob)
    aucs.append(score)

    print('AUC: ', score)
    RocCurveDisplay.from_predictions(
        y_test,
        y_prob,
        name=f"Fold {i+1}", ax=ax
    )
    
print(f"5-Fold平均AUC: {np.mean(aucs):.4f} ± {np.std(aucs)}")
plt.show()



# CA-19-9の値を予想するモデル作成

## IL2RAとプロテオミクスデータを結合したものを作成

In [ ]:
display(pro_data)
display(ca199)
display(il2ra)

In [ ]:
temp = pro_data.copy()
temp["IL2RA"] = il2ra["logvalue"]
display(temp["IL2RA"])
pro_data_il2ra = temp


## LASSO回帰

### 全プロテオミクス

In [ ]:
X = pro_data_il2ra
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)
r2_list = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #標準化
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    lasso_model = LassoCV(cv=5, random_state=123, n_jobs=-1)
    lasso_model.fit(X_train, y_train)
    
    r2_list.append(r2_score(y_test, lasso_model.predict(X_test)))

    # 3. 予測と評価
    y_pred = lasso_model.predict(X_test)
    print(f"Test R2 Score: {r2_score(y_test, y_pred):.3f}")
    print(f"Test MAE: {mean_absolute_error(y_test, y_pred):.3f}")
    
    # 4. どのタンパク質がCA19-9の予測に貢献したか確認
    y_pred = lasso_model.predict(X_test)
    
    
    print("=== モデル評価結果 ===")
    print(f"Alpha: {lasso_model.alpha_:.5f}")
    print(f"R2 Score: {r2_score(y_test, y_pred):.3f}")
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")

    # 残差プロットの描画
    residuals = y_test.values.flatten() - y_pred
    plt.figure(figsize=(8, 5))
    plt.scatter(y_pred, residuals, alpha=0.5, color='purple')
    plt.axhline(y=0, color='r', linestyle='--')
    plt.title("Residual Plot")
    plt.xlabel("Predicted Values")
    plt.ylabel("Residuals (Actual - Predicted)")
    plt.show()

print(f"平均R2スコア: {np.mean(r2_list):.3f}")

# # 各特徴量（タンパク質）の係数（重み）を取得
# coefs = pd.Series(lasso_model.coef_, index=X.columns)

# # 係数が 0 でない（＝LASSOによって選ばれた）タンパク質だけを抽出
# selected_features = coefs[coefs != 0].sort_values(key=abs, ascending=False)

# print(f"\n全 {len(X.columns)} 個中、{len(selected_features)} 個のタンパク質が選択されました。")

# # 上位20個をグラフで可視化
# plt.figure(figsize=(10, 6))
# selected_features.head(20).sort_values().plot(kind='barh', color='salmon')
# plt.title("Top 20 Proteins Predicting CA19-9 (LASSO Coefficients)")
# plt.xlabel("Coefficient Value (Weight)")
# plt.ylabel("Proteins")
# plt.tight_layout()
# plt.show()

# print("\n--- 選択された上位タンパク質の一覧 ---")
# print(selected_features.head(20))



### DEPlist+IL2RAのみ

In [ ]:
display(DEPlist)

In [ ]:
dep = pro_data.loc[:, pro_data.columns.isin(DEPlist.index)]

X = pd.concat([
    il2ra[['logvalue']],
    dep
], axis=1)

y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)
r2_list = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #標準化
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    lasso_model = LassoCV(cv=5, random_state=123, n_jobs=-1)
    lasso_model.fit(X_train, y_train)
    
    r2_list.append(r2_score(y_test, lasso_model.predict(X_test)))

    # 3. 予測と評価
    y_pred = lasso_model.predict(X_test)
    print(f"Test R2 Score: {r2_score(y_test, y_pred):.3f}")
    print(f"Test MAE: {mean_absolute_error(y_test, y_pred):.3f}")
    
    # 4. どのタンパク質がCA19-9の予測に貢献したか確認
    y_pred = lasso_model.predict(X_test)
    
    print("=== モデル評価結果 ===")
    print(f"Alpha: {lasso_model.alpha_:.5f}")
    print(f"R2 Score: {r2_score(y_test, y_pred):.3f}")
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")

        # 残差プロットの描画
    residuals = y_test.values.flatten() - y_pred
    plt.figure(figsize=(8, 5))
    plt.scatter(y_pred, residuals, alpha=0.5, color='purple')
    plt.axhline(y=0, color='r', linestyle='--')
    plt.title("Residual Plot")
    plt.xlabel("Predicted Values")
    plt.ylabel("Residuals (Actual - Predicted)")
    plt.show()

print(f"平均R2スコア: {np.mean(r2_list):.3f}")

# # 各特徴量（タンパク質）の係数（重み）を取得
# coefs = pd.Series(lasso_model.coef_, index=X.columns)

# # 係数が 0 でない（＝LASSOによって選ばれた）タンパク質だけを抽出
# selected_features = coefs[coefs != 0].sort_values(key=abs, ascending=False)

# print(f"\n全 {len(X.columns)} 個中、{len(selected_features)} 個のタンパク質が選択されました。")

# # 上位20個をグラフで可視化
# plt.figure(figsize=(10, 6))
# selected_features.head(20).sort_values().plot(kind='barh', color='salmon')
# plt.title("Top 20 Proteins Predicting CA19-9 (LASSO Coefficients)")
# plt.xlabel("Coefficient Value (Weight)")
# plt.ylabel("Proteins")
# plt.tight_layout()
# plt.show()

# print("\n--- 選択された上位タンパク質の一覧 ---")
# print(selected_features.head(20))



## ランダムフォレスト

### 全プロテオミクス

In [ ]:
X = pro_data_il2ra
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)
r2_list = []
mae_list = []
# fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 3]
    }
    
    model = RandomForestRegressor(
        criterion='absolute_error',
        random_state=123,
        n_jobs=-1 
    )
    
    cv_gen = KFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='r2',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    model.fit(X_train, y_train.values.ravel())

    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    r2_list.append(r2)
    mae_list.append(mae)
    

    print(f"Fold {i+1} - Test R2 Score: {r2:.3f} | Test MAE: {mae:.3f}")

print("\n=== 全体平均結果 ===")
print(f"平均R2スコア: {np.mean(r2_list):.3f}")
print(f"平均MAE: {np.mean(mae_list):.3f}")



### DEPlist+IL2RAのみ

In [ ]:
display(DEPlist)

In [ ]:
cols = list(DEPlist.index) + ["IL2RA"]
X = pro_data_il2ra.loc[:, pro_data.columns.isin(cols)]
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)
r2_list = []
mae_list = []
# fig, ax = plt.subplots()

for i, (train_index, test_index) in enumerate(kf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 20, 30,40],
        'min_samples_split': [2, 3, 4],
        'min_samples_leaf': [1, 2, 3]
    }
    
    model = RandomForestRegressor(
        criterion='absolute_error',
        random_state=123,
        n_jobs=-1 
    )
    
    cv_gen = KFold(n_splits=5, shuffle=True, random_state=123)
    
    modelcv = GridSearchCV(
        model, 
        param_grid=param_grid,
        scoring='r2',
        n_jobs=-1, 
        refit=True, 
        cv=cv_gen,
        verbose=0, 
        pre_dispatch='2*n_jobs', 
        error_score=np.nan, 
        return_train_score=False
    )
    
    # 実行
    model.fit(X_train, y_train.values.ravel())

    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    r2_list.append(r2)
    mae_list.append(mae)
    

    print(f"Fold {i+1} - Test R2 Score: {r2:.3f} | Test MAE: {mae:.3f}")

print("\n=== 全体平均結果 ===")
print(f"平均R2スコア: {np.mean(r2_list):.3f}")
print(f"平均MAE: {np.mean(mae_list):.3f}")



## 予測されたCA-19-9の値を見る

### 全プロテオミクス

In [ ]:
X = pro_data
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['CancerStatus'] = pro_sample.loc[tempI, 'CaCo_Status']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)


# 上位50サンプルを表示
print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {0: 'teal', 1: 'orange'}
labels = {0: 'Status 0 (Healthy)', 1: 'Status 1 (Cancer)'}

for status in [0, 1]:
    subset = comparison_df[comparison_df['CancerStatus'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("ALL Proteome : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# 3. 係数が0でない（選ばれた）タンパク質のみ表示
selected_proteins = importance_df[importance_df['Coefficient'] != 0]
print(len(selected_proteins))
display(selected_proteins)

In [ ]:
# selected proteinの詳細を見るためエクスポート
selected_proteins.to_csv("./Data/lasso_selectedProteins.csv", index=False)

In [ ]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# 1. パイプラインの定義 (標準化 + LassoCV)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

# 2. 交差検証の実行 (R^2スコアを算出)
# yは.values.ravel()で1次元配列に変換します
scores = cross_val_score(pipeline, X, y.values.ravel(), cv=5, scoring='r2')

print(f"各分割のR^2スコア: {scores}")
print(f"平均R^2: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")


### DEPlistのみ

In [ ]:
cols = list(DEPlist.index) + ["IL2RA"]
X = pro_data_il2ra.loc[:, pro_data.columns.isin(cols)]
y = ca199[["logvalue"]]

print(X.shape)
print(y.shape)

print(X.index[:5])
print(y.index[:5])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=123
)

tempI = X_test.index

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=123, n_jobs=-1))
])

pipeline.fit(X_train, y_train.values.ravel())


y_pred = pipeline.predict(X_test)

comparison_df = pd.DataFrame({
    'Actual_CA19_9': np.array(y_test).flatten(),
    'Predicted_CA19_9': np.array(y_pred).flatten()
}, index=tempI)
comparison_df['Abs_Diff'] = np.abs(comparison_df['Actual_CA19_9'] - comparison_df['Predicted_CA19_9'])
comparison_df['CancerStatus'] = pro_sample.loc[tempI, 'CaCo_Status']
comparison_df = comparison_df.sort_values(by='Abs_Diff', ascending=False)


# 上位50サンプルを表示
print("--- 各サンプルの実際のCA19-9 vs 予測されたCA19-9 ---")
print(comparison_df.head(50))

plt.figure(figsize=(6, 6))

# Status（0と1）でグループ分けしてプロット
colors = {0: 'teal', 1: 'orange'}
labels = {0: 'Status 0 (Healthy)', 1: 'Status 1 (Cancer)'}

for status in [0, 1]:
    subset = comparison_df[comparison_df['CancerStatus'] == status]
    plt.scatter(subset['Actual_CA19_9'], subset['Predicted_CA19_9'], 
                alpha=0.7, c=colors[status], label=labels[status])

# 完璧に予測できた場合の補助線（既存の処理）
max_val = max(comparison_df['Actual_CA19_9'].max(), comparison_df['Predicted_CA19_9'].max())
min_val = min(comparison_df['Actual_CA19_9'].min(), comparison_df['Predicted_CA19_9'].min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Prediction')

plt.title("DEPlist+IL2RA : Actual vs Predicted CA19-9 by Status")
plt.xlabel("Actual CA19-9")
plt.ylabel("Predicted CA19-9")
plt.legend()
plt.grid(True)
plt.show()

# 1. パイプラインからLassoの係数を取り出す
coef = pipeline.named_steps['lasso'].coef_
feature_names = X.columns

# 2. データフレームにまとめて、影響度順にソート
importance_df = pd.DataFrame({
    'Protein': feature_names,
    'Coefficient': coef
}).assign(Abs_Coef = lambda x: x['Coefficient'].abs()) \
  .sort_values(by='Abs_Coef', ascending=False)

# 3. 係数が0でない（選ばれた）タンパク質のみ表示
selected_proteins = importance_df[importance_df['Coefficient'] != 0]
print(len(selected_proteins))
display(selected_proteins)

# SHAP value for each models

In [ ]:
%pip install --force-reinstall --no-cache-dir "shap<0.46"

In [ ]:
import numpy as np
print(np.__version__)

In [ ]:
import shap


shap.initjs()

# フォント設定
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Liberation Sans", "DejaVu Sans"]


def calculate_cv_shap(data, sample, models, data_name):

    # DEPlistのタンパク質だけを抽出
    X = data.loc[:, data.columns.isin(DEPlist.index)]
    y = sample["Cancer"]

    # モデルと特徴量の順番を確認
    print(f"\n===== {data_name} =====")
    print("X shape:", X.shape)
    print("Number of models:", len(models))

    all_model_shap_values = []

    print("Calculating SHAP values...")

    # 5モデルすべてについてSHAPを計算
    for i, model in enumerate(models):

        explainer = shap.TreeExplainer(model)
        shap_values = explainer(X)

        # クラス1（癌）のSHAP value
        if len(shap_values.shape) == 3:
            shap_values_class1 = shap_values[:, :, 1]
        else:
            shap_values_class1 = shap_values

        all_model_shap_values.append(shap_values_class1)

        print(f"Fold {i+1} completed")

    # --------------------------------
    # 5モデルのSHAP valueを平均
    # --------------------------------

    mean_shap_values = all_model_shap_values[0]

    combined_values = np.zeros_like(mean_shap_values.values)

    for sv in all_model_shap_values:
        combined_values += sv.values

    mean_shap_values.values = (
        combined_values / len(models)
    )

    # --------------------------------
    # Mean Absolute SHAP Value
    # --------------------------------

    shap_v = np.abs(
        mean_shap_values.values
    ).mean(axis=0)

    feature_names = X.columns.tolist()

    shap_importance_df = pd.DataFrame({
        "Feature": feature_names,
        "SHAP_Importance": shap_v
    })

    sorted_shap_importance_df = (
        shap_importance_df
        .sort_values(
            by="SHAP_Importance",
            ascending=False
        )
        .reset_index(drop=True)
    )

    print("\n--- SHAP Based Feature Importance ---")
    display(sorted_shap_importance_df.head(15))

    # --------------------------------
    # SHAP Summary Plot
    # --------------------------------

    plt.figure(figsize=(7, 5))

    shap.summary_plot(
        mean_shap_values,
        X,
        max_display=15,
        show=False
    )

    plt.title(
        f"SHAP Summary Plot ({data_name})",
        fontsize=13,
        pad=15
    )

    plt.tight_layout()

    plt.savefig(
        f"shap_summary_plot_cv5_{data_name}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    return sorted_shap_importance_df, mean_shap_values

In [ ]:
shap_importance_main, shap_values_main = calculate_cv_shap(
    data=pro_data,
    sample=pro_sample,
    models=modelsets_HER_DEP,
    data_name="pro_data"
)

# CA19-9とメタデータの項目の関連

In [ ]:
display(pro_sample)